In [1]:
import pandas as pd
import numpy as np

## Yield Prediction using Gradient Boosting and PyTorch

### 1. Data Loading and Initial Exploration

First, we'll load the dataset and take a look at its structure, data types, and a few sample rows.

In [2]:
# Load the dataset
try:
    df = pd.read_csv('/content/production_unified_imputed.csv')
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Error: The file '/content/production_unified_imputed.csv' was not found. Please ensure it's uploaded correctly.")
    df = None # Set df to None to avoid further errors

# Display the first 5 rows of the DataFrame if loaded successfully
if df is not None:
    print("\nFirst 5 rows of the dataset:")
    display(df.head())

    # Display basic information about the DataFrame
    print("\nDataFrame Info:")
    df.info()

Dataset loaded successfully.

First 5 rows of the dataset:


,crop,year,state,district,season,area,production,yield,data_source,annual_rainfall,fertilizer,pesticide,crop_type,area_unit,production_unit,yield_unit
0,wheat,2014,uttar pradesh,ghaziabad,Rabi,26619.0,86645.0,3.255,area_production,1247.199725,1.248187e+06,2438.165184,cereals,Hectare,Tonnes,Tonnes/Hectare
1,urad,2014,uttar pradesh,ghaziabad,Summer,22.0,12.0,0.545,area_production,1247.339375,1.235179e+06,2420.293698,pulses,Hectare,Tonnes,Tonnes/Hectare
2,urad,2014,uttar pradesh,ghaziabad,Kharif,19.0,10.0,0.526,area_production,1247.339396,1.235177e+06,2420.290923,pulses,Hectare,Tonnes,Tonnes/Hectare
3,sugarcane,2014,uttar pradesh,ghaziabad,Kharif,11136.0,750076.0,67.356,area_production,1247.223783,1.245441e+06,2437.372883,sugar,Hectare,Tonnes,Tonnes/Hectare
4,rice,2014,uttar pradesh,ghaziabad,Kharif,8650.0,23727.0,2.743,area_production,1247.278389,1.240637e+06,2428.557661,cereals,Hectare,Tonnes,Tonnes/Hectare



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440962 entries, 0 to 440961
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   crop             440962 non-null  object 
 1   year             440962 non-null  int64  
 2   state            440962 non-null  object 
 3   district         440962 non-null  object 
 4   season           440962 non-null  object 
 5   area             440962 non-null  float64
 6   production       440962 non-null  float64
 7   yield            440962 non-null  float64
 8   data_source      440962 non-null  object 
 9   annual_rainfall  440962 non-null  float64
 10  fertilizer       440962 non-null  float64
 11  pesticide        440962 non-null  float64
 12  crop_type        440962 non-null  object 
 13  area_unit        440962 non-null  object 
 14  production_unit  440962 non-null  object 
 15  yield_unit       440962 non-null  object 
dtypes: float64(6), int64(

### 2. Data Preprocessing

Now, let's prepare the data for modeling. This typically involves:
- Identifying categorical and numerical features.
- Encoding categorical features using techniques like one-hot encoding.
- Splitting the data into training and testing sets.

In [3]:
# Identify categorical and numerical features
categorical_features = df.select_dtypes(include=['object']).columns.tolist()
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# The target variable is 'yield', so remove it from features if present
if 'yield' in numerical_features:
    numerical_features.remove('yield')

print(f"Categorical features: {categorical_features}")
print(f"Numerical features (excluding target 'yield'): {numerical_features}")

# One-hot encode categorical features
df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=True)

print("\nDataFrame after one-hot encoding:")
display(df_encoded.head())
print(f"New shape of DataFrame: {df_encoded.shape}")

Categorical features: ['crop', 'state', 'district', 'season', 'data_source', 'crop_type', 'area_unit', 'production_unit', 'yield_unit']
Numerical features (excluding target 'yield'): ['year', 'area', 'production', 'annual_rainfall', 'fertilizer', 'pesticide']

DataFrame after one-hot encoding:


,year,area,production,yield,annual_rainfall,fertilizer,pesticide,crop_arcanut (processed),crop_arecanut,crop_arhar/tur,...,data_source_des_district,crop_type_drugs and narcotics,crop_type_fiber crops,crop_type_fruits,crop_type_oilseeds,crop_type_plantation crops,crop_type_pulses,crop_type_spices,crop_type_sugar,crop_type_vegetable
0,2014,26619.0,86645.0,3.255,1247.199725,1.248187e+06,2438.165184,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,2014,22.0,12.0,0.545,1247.339375,1.235179e+06,2420.293698,False,False,False,...,False,False,False,False,False,False,True,False,False,False
2,2014,19.0,10.0,0.526,1247.339396,1.235177e+06,2420.290923,False,False,False,...,False,False,False,False,False,False,True,False,False,False
3,2014,11136.0,750076.0,67.356,1247.223783,1.245441e+06,2437.372883,False,False,False,...,False,False,False,False,False,False,False,False,True,False
4,2014,8650.0,23727.0,2.743,1247.278389,1.240637e+06,2428.557661,False,False,False,...,False,False,False,False,False,False,False,False,False,False


New shape of DataFrame: (440962, 926)


### 3. Splitting Data into Training and Testing Sets

Now, we'll define our features (X) and target variable (y), and then split them into training and testing sets to evaluate our models effectively.

In [4]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
X = df_encoded.drop('yield', axis=1)
y = df_encoded['yield']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (352769, 925)
Shape of X_test: (88193, 925)
Shape of y_train: (352769,)
Shape of y_test: (88193,)


### 4. Preparing Data for PyTorch

To use PyTorch, we need to convert our numpy arrays (from pandas DataFrames/Series) into PyTorch tensors. We'll also create `DataLoader` objects for efficient batch processing during training.

In [5]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Convert to NumPy arrays first (if not already)
X_train_np = X_train.values.astype(np.float32)
X_test_np = X_test.values.astype(np.float32)
y_train_np = y_train.values.astype(np.float32)
y_test_np = y_test.values.astype(np.float32)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_np)
X_test_tensor = torch.tensor(X_test_np)
y_train_tensor = torch.tensor(y_train_np).unsqueeze(1) # Unsqueeze for a single output feature
y_test_tensor = torch.tensor(y_test_np).unsqueeze(1) # Unsqueeze for a single output feature

print(f"X_train_tensor shape: {X_train_tensor.shape}")
print(f"y_train_tensor shape: {y_train_tensor.shape}")

# Create TensorDatasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create DataLoaders
batch_size = 64 # You can adjust this batch size
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\nNumber of training batches: {len(train_loader)}")
print(f"Number of testing batches: {len(test_loader)}")

X_train_tensor shape: torch.Size([352769, 925])
y_train_tensor shape: torch.Size([352769, 1])

Number of training batches: 5513
Number of testing batches: 1379


### 5. Define PyTorch Model Architecture

Now we will define a simple Multi-Layer Perceptron (MLP) using PyTorch's `nn.Module` for our yield prediction task. Since this is a regression problem, the output layer will have a single neuron without an activation function, suitable for predicting continuous values.

In [6]:
import torch.nn as nn
import torch.optim as optim

# Define the neural network architecture
class YieldPredictor(nn.Module):
    def __init__(self, input_size):
        super(YieldPredictor, self).__init__()
        self.fc1 = nn.Linear(input_size, 256) # First fully connected layer
        self.relu1 = nn.ReLU()                # ReLU activation
        self.dropout1 = nn.Dropout(0.3)       # Dropout for regularization
        self.fc2 = nn.Linear(256, 128)        # Second fully connected layer
        self.relu2 = nn.ReLU()                # ReLU activation
        self.dropout2 = nn.Dropout(0.3)       # Dropout for regularization
        self.fc3 = nn.Linear(128, 64)         # Third fully connected layer
        self.relu3 = nn.ReLU()                # ReLU activation
        self.dropout3 = nn.Dropout(0.2)       # Dropout for regularization
        self.fc4 = nn.Linear(64, 1)           # Output layer (1 neuron for regression)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        x = self.relu3(x)
        x = self.dropout3(x)
        x = self.fc4(x)
        return x

# Instantiate the model
input_size = X_train_tensor.shape[1]
model = YieldPredictor(input_size)

# Define loss function and optimizer
criterion = nn.MSELoss() # Mean Squared Error for regression
optimizer = optim.Adam(model.parameters(), lr=0.001) # Adam optimizer

print(model)
print(f"\nLoss function: {criterion}")
print(f"Optimizer: {optimizer}")

YieldPredictor(
  (fc1): Linear(in_features=925, out_features=256, bias=True)
  (relu1): ReLU()
  (dropout1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (relu2): ReLU()
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=128, out_features=64, bias=True)
  (relu3): ReLU()
  (dropout3): Dropout(p=0.2, inplace=False)
  (fc4): Linear(in_features=64, out_features=1, bias=True)
)

Loss function: MSELoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


### 6. Train the PyTorch Model

We will now train the defined `YieldPredictor` model using our `train_loader`. We'll monitor the training loss and evaluate the model's performance on the test set after training.

In [ ]:
# Check if CUDA is available and use GPU if it is
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Using device: {device}")

# Training loop
num_epochs = 10  # You can adjust the number of epochs

for epoch in range(num_epochs):
    model.train()  # Set model to training mode
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()  # Zero the parameter gradients
        outputs = model(inputs)  # Forward pass
        loss = criterion(outputs, labels)  # Calculate loss
        loss.backward()  # Backward pass
        optimizer.step()  # Optimize

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")

print("\nTraining finished!")

Using device: cpu
Epoch 1/10, Loss: 73900670.0026
Epoch 2/10, Loss: 12959.2578


### 7. Evaluate the PyTorch Model

After training, we will evaluate the model's performance on the unseen test dataset using metrics like Mean Squared Error (MSE) and Root Mean Squared Error (RMSE).

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

model.eval()  # Set model to evaluation mode
predictions = []
true_labels = []

with torch.no_grad():  # Disable gradient calculation during evaluation
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        predictions.extend(outputs.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

predictions = np.array(predictions).flatten()
true_labels = np.array(true_labels).flatten()

mse = mean_squared_error(true_labels, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(true_labels, predictions)

print(f"\nModel Evaluation on Test Set:")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R-squared (R2): {r2:.4f}")